# Store OpenLineage events in PostgreSQL

Sets up Egeria's PostgreSQL OpenLineage log store.  It catalogs the `open_lineage_log_store` schema of the `egeria_freshstart` database on the shared PostgreSQL server, and then:

* makes it the catalog target of the **OpenLineagePostgresPublisher** integration connector, which creates the schema and its tables if they do not exist, stores every OpenLineage event Egeria receives or generates, and each hour deletes the records older than its `daysToCleanUp` configuration property (30 days);
* links it, as the `openLineageLogStore` action target, to the three Lovelace governance action types that analyse OpenLineage events (*profile-open-lineage-runs*, *refine-open-lineage-data-scope* and *summarise-open-lineage-data-quality*), so they read their time window from the database rather than re-reading every file in the file-based log store;
* removes the **OpenLineageFilePublisher** from the Open Lineage integration group, so the events are no longer also saved as files in `logs/openlineage` (nothing deletes old files from there).

Each step checks whether it has already been done, so the notebook is safe to run again.  The database connection uses the same database user as Egeria's PostgreSQL repository (the `PostgreSQLRepository` collection in `secrets/egeria-servers.omsecrets`), which can create schemas in the repository database.  Override the defaults with `OPENLINEAGE_LOG_STORE_HOST`, `OPENLINEAGE_LOG_STORE_PORT`, `OPENLINEAGE_LOG_STORE_DATABASE` and `OPENLINEAGE_LOG_STORE_SCHEMA`.

The Portal's bootstrap monitor runs this notebook after a demo reset (see `_batch.json`).

In [ ]:
import os

from pyegeria import ActionAuthor, AssetMaker, AutomatedCuration, ClassificationExplorer, MetadataExpert, RuntimeManager

view_server = os.environ.get("EGERIA_VIEW_SERVER", "fs-view-server")
url = os.environ.get("EGERIA_VIEW_SERVER_URL", "https://freshstart-egeria-main:8443")
user_id = os.environ.get("EGERIA_USER", "bootstrap")
user_pwd = os.environ.get("EGERIA_USER_PASSWORD", "secret")
integration_daemon_name = os.environ.get("EGERIA_INTEGRATION_DAEMON", "fs-integration-daemon")

host_identifier = os.environ.get("OPENLINEAGE_LOG_STORE_HOST", "host.docker.internal")
port_number     = os.environ.get("OPENLINEAGE_LOG_STORE_PORT", "5442")
database_name   = os.environ.get("OPENLINEAGE_LOG_STORE_DATABASE", "egeria_freshstart")
schema_name     = os.environ.get("OPENLINEAGE_LOG_STORE_SCHEMA", "open_lineage_log_store")
server_name     = "egeria-shared-postgres"

POSTGRES_SCHEMA_TEMPLATE_GUID = "82a5417c-d882-4271-8444-4c6a996a8bfc"    # PostgreSQL Relational Database Schema template
PUBLISHER_GUID                = "a388de80-081e-4303-894c-b3dc54147fcc"    # OpenLineagePostgresPublisher integration connector
PUBLISHER_NAME                = "OpenLineagePostgresPublisher"
FILE_PUBLISHER_GUID           = "6271b678-7d22-4cdf-87b1-45b366beaf4e"    # OpenLineageFilePublisher integration connector
LOG_STORE_TARGET_NAME         = "openLineageLogStore"
LOVELACE_ACTION_TYPES         = ["EgeriaGovernance::profile-open-lineage-runs",
                                 "EgeriaGovernance::refine-open-lineage-data-scope",
                                 "EgeriaGovernance::summarise-open-lineage-data-quality"]

classification_explorer = ClassificationExplorer(view_server, url, user_id, user_pwd)
asset_maker = AssetMaker(view_server, url, user_id, user_pwd)
automated_curation = AutomatedCuration(view_server, url, user_id, user_pwd)
action_author = ActionAuthor(view_server, url, user_id, user_pwd)
runtime_manager = RuntimeManager(view_server, url, user_id, user_pwd)
metadata_expert = MetadataExpert(view_server, url, user_id, user_pwd)

token = classification_explorer.create_egeria_bearer_token(user_id, user_pwd)
for client in (asset_maker, automated_curation, action_author, runtime_manager, metadata_expert):
    client.set_bearer_token(token)


def guid_of(qualified_name):
    """Return the GUID of the element with this qualified name, or None."""
    result = classification_explorer.get_element_guid_by_unique_name(qualified_name)
    return result if isinstance(result, str) and result != "No elements found" else None


def collect_guids(obj, found=None):
    """Walk a pyegeria response and gather every 'guid' value in it."""
    if found is None:
        found = set()
    if isinstance(obj, dict):
        for key, value in obj.items():
            if key == "guid" and isinstance(value, str):
                found.add(value)
            else:
                collect_guids(value, found)
    elif isinstance(obj, list):
        for value in obj:
            collect_guids(value, found)
    return found


def related_guids(element_guid, relationship_type, start_at_end):
    """GUIDs of the elements linked to element_guid by relationship_type (empty when there are none)."""
    result = classification_explorer.get_related_elements(element_guid, relationship_type, start_at_end=start_at_end)
    return collect_guids(result) if isinstance(result, list) else set()


# 1. The schema asset

schema_qualified_name = f"PostgreSQL Relational Database Schema::{server_name}::{database_name}.{schema_name}"
schema_guid = guid_of(schema_qualified_name)
if schema_guid is None:
    schema_guid = asset_maker.create_asset_from_template({
        "class": "TemplateRequestBody",
        "templateGUID": POSTGRES_SCHEMA_TEMPLATE_GUID,
        "isOwnAnchor": True,
        "placeholderPropertyValues": {
            "hostIdentifier": host_identifier,
            "portNumber": port_number,
            "serverName": server_name,
            "versionIdentifier": "1.0",
            "databaseName": database_name,
            "schemaName": schema_name,
            "schemaDescription": "Egeria's OpenLineage log store: every OpenLineage event Egeria receives or generates, "
                                 "with a digest of each run, the datasets it read and wrote, and its data quality "
                                 "assertions and tests.",
            "secretsStorePathName": "secrets/egeria-servers.omsecrets",
            "secretsCollectionName": "PostgreSQLRepository"
        }
    })
    print(f"Created {schema_qualified_name}: {schema_guid}")
else:
    print(f"Found {schema_qualified_name}: {schema_guid}")

if not isinstance(schema_guid, str) or schema_guid.startswith("No "):
    raise RuntimeError(f"The schema asset could not be created: {schema_guid}")

# 2. The catalog target of the publisher.  CatalogTarget: end 1 is the integration connector, end 2 is the element.

if PUBLISHER_GUID in related_guids(schema_guid, "CatalogTarget", start_at_end=2):
    print(f"{schema_qualified_name} is already a catalog target of {PUBLISHER_NAME}")
else:
    automated_curation.add_catalog_target(PUBLISHER_GUID, schema_guid, LOG_STORE_TARGET_NAME)
    print(f"Added {schema_qualified_name} as the {LOG_STORE_TARGET_NAME} catalog target of {PUBLISHER_NAME}")

# 3. The action target of the Lovelace governance action types.
#    TargetForGovernanceAction: end 1 is the governance action type, end 2 is the target.

linked_action_types = related_guids(schema_guid, "TargetForGovernanceAction", start_at_end=2)
for action_type_name in LOVELACE_ACTION_TYPES:
    action_type_guid = guid_of(action_type_name)
    if action_type_guid is None:
        print(f"{action_type_name} is not in Egeria - is the OpenLineage content pack loaded?")
    elif action_type_guid in linked_action_types:
        print(f"{action_type_name} already uses the log store")
    else:
        action_author.link_target_for_governance_action(action_type_guid, schema_guid, {
            "class": "NewRelationshipRequestBody",
            "properties": {
                "class": "TargetForGovernanceActionProperties",
                "actionTargetName": LOG_STORE_TARGET_NAME
            }
        })
        print(f"Linked {action_type_name} to the log store")

# 4. Stop saving the events as files in logs/openlineage - the database replaces that file store, which nothing
#    ever cleans up.  The OpenLineageFilePublisher is registered with the Open Lineage integration group by a content
#    pack (the OpenLineageFileStoreContentPack, which this deployment does not load, or the OpenLineageContentPack
#    before Egeria 6.3), so the RegisteredIntegrationConnector relationship is removed in the name of that content
#    pack.  The integration daemon stops the connector when it next refreshes the group.

registrations = classification_explorer.get_related_elements(FILE_PUBLISHER_GUID, "RegisteredIntegrationConnector",
                                                             start_at_end=2)
if not isinstance(registrations, list) or not registrations:
    print("The OpenLineageFilePublisher is not registered with an integration group - events are not saved as files")
for registration in registrations if isinstance(registrations, list) else []:
    relationship_header = (registration.get("relatedBy") or {}).get("relationshipHeader") or {}
    relationship_guid = relationship_header.get("guid")
    if relationship_guid is None:
        continue
    origin = relationship_header.get("origin") or {}
    delete_body = {"class": "DeleteRelationshipRequestBody"}
    if origin.get("homeMetadataCollectionId"):
        delete_body["externalSourceGUID"] = origin["homeMetadataCollectionId"]
        delete_body["externalSourceName"] = origin.get("homeMetadataCollectionName")
    metadata_expert.delete_related_elements(relationship_guid, delete_body)
    print(f"Removed the OpenLineageFilePublisher from its integration group ({relationship_guid}) - "
          f"events are no longer saved as files")

# 5. Refresh the publisher so it sets up the schema now rather than at its next hourly refresh.

try:
    servers = runtime_manager.get_servers_by_name(filter_string=integration_daemon_name)
    daemon_guids = [s["elementHeader"]["guid"] for s in servers if isinstance(s, dict)] if isinstance(servers, list) else []
    for daemon_guid in daemon_guids:
        runtime_manager.refresh_integration_connector(connector_name=PUBLISHER_NAME, server_guid=daemon_guid)
    print(f"Refreshed {PUBLISHER_NAME} in {integration_daemon_name}" if daemon_guids
          else f"{integration_daemon_name} not found - {PUBLISHER_NAME} picks up its catalog target at its next refresh")
except Exception as error:
    print(f"The refresh did not complete ({error}) - {PUBLISHER_NAME} picks up its catalog target at its next refresh")
